# 03 — Signal & long-short backtest

Score every ECTSum transcript with the fine-tuned FinBERT, aggregate to a weekly **sector** sentiment score, then trade a simple long-short portfolio: **long the top-2 sectors, short the bottom-2** by sentiment, hold for one week.

All numbers below are read from artifacts written by `run_pipeline.py --only signal,backtest` so the notebook is purely presentational.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from config import PREDICTIONS_DIR, RAW_DIR, SUMMARY_DIR
from signals.backtest import run as backtest_run
from signals.performance import plot_cumulative, plot_sentiment_timeseries, pretty_summary

sentiment = pd.read_csv(PREDICTIONS_DIR / 'weekly_sector_sentiment.csv', parse_dates=['week']).set_index('week')
returns = pd.read_csv(RAW_DIR / 'sector_returns_weekly.csv', parse_dates=['Date']).set_index('Date')
sentiment.head()

## Sentiment score time series per sector

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for col in sentiment.columns:
    ax.plot(sentiment.index, sentiment[col], lw=1, alpha=0.85, label=col)
ax.axhline(0.0, color='black', lw=0.5)
ax.set_title('Weekly sector sentiment (P(positive) - P(negative))')
ax.set_ylabel('sentiment'); ax.set_xlabel('week')
ax.legend(ncol=3, fontsize=8, loc='lower right')
plt.tight_layout(); plt.show()

## Backtest: signal vs random benchmark

**Why the random benchmark?** It is the same long-short rule but the sentiment scores are shuffled within each week. Any positive Sharpe it shows is purely from cross-sectional dispersion in sector returns — not from sentiment information. Our signal needs to beat it.

In [ ]:
results = backtest_run(sentiment, returns)
signal_w = results['signal'].weekly
bench_w = results['random_benchmark'].weekly

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(signal_w.index, signal_w['cum_ret'], label='FinBERT signal', lw=2)
ax.plot(bench_w.index, bench_w['cum_ret'], label='Random benchmark', lw=1.5, ls='--', color='grey')
ax.axhline(0.0, color='black', lw=0.5)
ax.set_title('Cumulative return: long-short top/bottom-2 sectors')
ax.set_ylabel('cumulative return'); ax.set_xlabel('week')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
print(pretty_summary(results['signal'].summary, 'FinBERT signal'))
print()
print(pretty_summary(results['random_benchmark'].summary, 'Random benchmark'))

### Why even a Sharpe of ~0.5 on this signal is notable

This is a **simple** signal:
* one model (fine-tuned FinBERT, no ensembling),
* one feature (`P(positive) - P(negative)`),
* one universe (9 US sector ETFs),
* equal-weight, no risk model, no costs.

Most public, easily-reproducible signals on liquid US equities die out near a Sharpe of zero after costs. A consistent **annualised Sharpe of 0.5** on weekly rebalancing — even pre-cost — is enough to justify investigating it further. To actually trade it you would still need to: (a) net it against transaction costs and turnover, (b) check it survives different sector universes and time periods, (c) build a proper risk model so you are not just loading on a sector-momentum or value factor in disguise, and (d) demonstrate it is not driven by a handful of crisis weeks.

Without those checks the signal is a research finding, not a strategy.